# 0150 · CLIMA HISTÓRICO DE BOGOTÁ (variables externas para la hipótesis H2)

**Qué hace:** descarga de Open-Meteo (API gratuita, sin clave para uso académico) la lluvia, temperatura y humedad **hora a hora** de enero a agosto de 2026 en 10 puntos de Bogotá, construye variables de clima y las cruza con `BASE_FINAL_BOGOTA.parquet` por fecha y hora de envío.

**Dónde correrlo:** en el Databricks de John (donde están los datos). Si Databricks no tiene salida a internet, correr las celdas 1 a 6 en cualquier computador (Jupyter / Colab) con `PATH_DATABRICKS` apuntando a una carpeta local, y subir los dos Parquet de la carpeta `EXTERNAS/` al Volume.

**Orden en el proyecto:** 0100 → 0200 → 0300 → **0150 (este)** → 0400.
La celda 7 necesita que ya exista `BASE_FINAL_BOGOTA.parquet` (la crea el 0400, celda 6).

**Fuente:** Open-Meteo Historical Weather API (`archive-api.open-meteo.com`), modelo *best match* (IFS 9 km + ERA5 / ERA5-Land). Citar en el documento como fuente de datos externos.

In [ ]:
# CELDA 1 · Librerías
%pip install duckdb requests

In [ ]:
# CELDA 2 · Rutas y parámetros
import os, time, datetime
import requests
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"   # o una carpeta local si se corre fuera de Databricks

CARPETA_EXTERNAS        = f"{PATH_DATABRICKS}EXTERNAS/"
RUTA_CLIMA_PUNTOS       = f"{CARPETA_EXTERNAS}CLIMA_BOGOTA_PUNTOS_HORARIO.parquet"   # una fila por punto y hora
RUTA_CLIMA_CIUDAD       = f"{CARPETA_EXTERNAS}CLIMA_BOGOTA_CIUDAD_HORARIO.parquet"   # una fila por hora (promedio ciudad)
RUTA_BASE_FINAL         = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/BASE_FINAL_BOGOTA.parquet"
RUTA_BASE_FINAL_CLIMA   = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/BASE_FINAL_BOGOTA_CLIMA.parquet"

FECHA_INICIO = "2026-01-01"
FECHA_FIN    = "2026-08-31"
os.makedirs(CARPETA_EXTERNAS, exist_ok=True)
print("Destino:", CARPETA_EXTERNAS)

In [ ]:
# CELDA 3 · Puntos de referencia en Bogotá (centro aproximado de 10 localidades)
# Cubren norte, occidente, centro, sur y suroriente para captar que la lluvia no cae igual en toda la ciudad.
PUNTOS = pd.DataFrame([
    ("USAQUEN",          4.703, -74.031),
    ("SUBA",             4.741, -74.084),
    ("ENGATIVA",         4.705, -74.113),
    ("FONTIBON",         4.673, -74.146),
    ("CHAPINERO_CENTRO", 4.634, -74.066),
    ("KENNEDY",          4.630, -74.152),
    ("BOSA",             4.618, -74.190),
    ("CIUDAD_BOLIVAR",   4.560, -74.150),
    ("SAN_CRISTOBAL",    4.566, -74.087),
    ("USME",             4.505, -74.113),
], columns=["PUNTO", "LAT", "LON"])
display(PUNTOS)

In [ ]:
# CELDA 4 · Descarga desde Open-Meteo (una sola petición para los 10 puntos)
URL = "https://archive-api.open-meteo.com/v1/archive"
VARIABLES = ["precipitation", "rain", "temperature_2m", "relative_humidity_2m", "weather_code"]

params = {
    "latitude":  ",".join(str(x) for x in PUNTOS["LAT"]),
    "longitude": ",".join(str(x) for x in PUNTOS["LON"]),
    "start_date": FECHA_INICIO,
    "end_date":   FECHA_FIN,
    "hourly": ",".join(VARIABLES),
    "timezone": "America/Bogota",   # misma hora local que G_FECHA_ENVIO
}

def descargar(params, intentos=4):
    for i in range(intentos):
        r = requests.get(URL, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        print(f"Intento {i+1}: HTTP {r.status_code} -> {r.text[:200]}")
        time.sleep(10 * (i + 1))
    raise RuntimeError("No se pudo descargar el clima. ¿Databricks tiene salida a internet?")

respuesta = descargar(params)
if isinstance(respuesta, dict):        # con un solo punto la API devuelve un objeto, no una lista
    respuesta = [respuesta]

tablas = []
for punto, bloque in zip(PUNTOS["PUNTO"], respuesta):
    h = pd.DataFrame(bloque["hourly"])
    h["PUNTO"] = punto
    tablas.append(h)
df_puntos = pd.concat(tablas, ignore_index=True)
df_puntos["FECHA_HORA"] = pd.to_datetime(df_puntos["time"])
df_puntos = df_puntos.drop(columns="time").rename(columns={
    "precipitation": "PRECIPITACION_MM", "rain": "LLUVIA_MM", "temperature_2m": "TEMPERATURA_C",
    "relative_humidity_2m": "HUMEDAD_PCT", "weather_code": "CODIGO_CLIMA"})

df_puntos.to_parquet(RUTA_CLIMA_PUNTOS, index=False)
print(f"{len(df_puntos):,} filas ({df_puntos['PUNTO'].nunique()} puntos × {df_puntos['FECHA_HORA'].nunique():,} horas)")
print("Guardado en:", RUTA_CLIMA_PUNTOS)

In [ ]:
# CELDA 5 · Control de calidad y gráfica de lluvia diaria
print("Nulos por variable:"); display(df_puntos.isna().sum())

diaria = (df_puntos.assign(FECHA=df_puntos["FECHA_HORA"].dt.date)
          .groupby(["FECHA", "PUNTO"])["PRECIPITACION_MM"].sum().unstack())
fig, ax = plt.subplots(figsize=(14, 4))
diaria.mean(axis=1).plot(ax=ax)
ax.set_title("Precipitación diaria promedio en Bogotá (mm) · enero–agosto 2026")
ax.set_ylabel("mm/día"); plt.tight_layout(); plt.show()

print("Diferencia entre puntos (lluvia total del periodo, mm):")
display(diaria.sum().sort_values(ascending=False).round(0))

In [ ]:
# CELDA 6 · Variables de clima a nivel ciudad (una fila por hora)
# Solo usan información disponible en el momento del envío (horas anteriores), para no filtrar el futuro.
ciudad = (df_puntos.groupby("FECHA_HORA")
          .agg(PRECIP_MM_PROM=("PRECIPITACION_MM", "mean"),
               PRECIP_MM_MAX=("PRECIPITACION_MM", "max"),
               PUNTOS_CON_LLUVIA=("PRECIPITACION_MM", lambda s: int((s >= 0.5).sum())),
               TEMPERATURA_C=("TEMPERATURA_C", "mean"),
               HUMEDAD_PCT=("HUMEDAD_PCT", "mean"))
          .sort_index())

ciudad["LLUEVE_HORA"]      = (ciudad["PRECIP_MM_PROM"] >= 0.5).astype(int)
ciudad["LLUVIA_3H_PREV"]   = ciudad["PRECIP_MM_PROM"].shift(1).rolling(3, min_periods=1).sum()
ciudad["LLUVIA_24H_PREV"]  = ciudad["PRECIP_MM_PROM"].shift(1).rolling(24, min_periods=1).sum()
ciudad["LLUVIA_72H_PREV"]  = ciudad["PRECIP_MM_PROM"].shift(1).rolling(72, min_periods=1).sum()
ciudad = ciudad.fillna(0).reset_index()

ciudad.to_parquet(RUTA_CLIMA_CIUDAD, index=False)
print(f"{len(ciudad):,} horas | horas con lluvia: {ciudad['LLUEVE_HORA'].mean():.1%}")
print("Guardado en:", RUTA_CLIMA_CIUDAD)
display(ciudad.head())

## Cruce con la base final (requiere haber corrido el 0400 hasta la celda 6)

Se une cada envío con el clima de la **hora en que salió** (`date_trunc('hour', G_FECHA_ENVIO)`). El resultado, `BASE_FINAL_BOGOTA_CLIMA.parquet`, es la **versión "internas + externas"** de la base que pide la hipótesis H2; la `BASE_FINAL_BOGOTA.parquet` sigue siendo la versión "solo internas".

In [ ]:
# CELDA 7 · Cruce envíos × clima y primera evidencia para H2
con = duckdb.connect()
con.execute(f"""
COPY (
    SELECT B.*,
           C.PRECIP_MM_PROM, C.PRECIP_MM_MAX, C.PUNTOS_CON_LLUVIA, C.LLUEVE_HORA,
           C.LLUVIA_3H_PREV, C.LLUVIA_24H_PREV, C.LLUVIA_72H_PREV,
           C.TEMPERATURA_C, C.HUMEDAD_PCT
    FROM read_parquet('{RUTA_BASE_FINAL}') B
    LEFT JOIN read_parquet('{RUTA_CLIMA_CIUDAD}') C
      ON date_trunc('hour', B.G_FECHA_ENVIO) = C.FECHA_HORA
) TO '{RUTA_BASE_FINAL_CLIMA}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.close()

df_h2 = duckdb.sql(f"""
    SELECT CASE WHEN LLUVIA_24H_PREV >= 5 THEN 'Llovió (>=5 mm en 24 h previas)' ELSE 'Seco' END AS CONDICION,
           COUNT(*) AS ENVIOS,
           ROUND(100 * AVG(INCUMPLIMIENTO_SLA), 2) AS TASA_INCUMPLIMIENTO_PCT,
           ROUND(MEDIAN(TIEMPO_ENTREGA_HORAS), 1)  AS TIEMPO_MEDIANO_H
    FROM read_parquet('{RUTA_BASE_FINAL_CLIMA}')
    GROUP BY 1 ORDER BY 1
""").df()
display(df_h2)
sin_clima = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{RUTA_BASE_FINAL_CLIMA}') WHERE PRECIP_MM_PROM IS NULL").fetchone()[0]
print(f"Envíos sin clima asignado: {sin_clima:,} (deberían ser 0 o muy pocos)")
print("Guardado en:", RUTA_BASE_FINAL_CLIMA)